In [11]:
import uuid
print(uuid.uuid1())
# print(uuid.uuid2())
# print(uuid.uuid3())/
print(type(str(uuid.uuid4())))

c5e68fc7-5602-11f1-874d-cdca6efea4e1
<class 'str'>


In [3]:
!pip install wikipedia

  Using cached wikipedia-1.4.0.tar.gz (27 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=378e070a0115258c9d99d1810038a1f4e9460b089d4496ec7232f982c041654e
  Stored in directory: c:\users\prath\appdata\local\pip\cache\wheels\8f\ab\cb\45ccc40522d3a1c41e1d2ad53b8f33a62f394011ec38cd71c6
Successfully built wikipedia


In [12]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


from tavily import TavilyClient
import wikipedia
import arxiv


# =========================
# STATE
# =========================

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

    arxiv_results: list
    wikipedia_results: list
    tavily_results: list


# =========================
# TOOLS
# =========================

@tool
def arxiv_tool(query: str):
    """
    Search research papers from arxiv
    """

    client = arxiv.Client(
        page_size=3,
        delay_seconds=3,
        num_retries=3
    )

    search = arxiv.Search(
        query=query,
        max_results=3
    )

    papers = []

    for result in client.results(search):

        papers.append({
            "title": result.title,
            "summary": result.summary[:300],
            "pdf_url": result.pdf_url
        })

    return papers


@tool
def wikipedia_tool(query: str):
    """
    Search wikipedia information
    """

    try:
        summary = wikipedia.summary(query, sentences=3)

        return {
            "query": query,
            "summary": summary
        }

    except Exception as e:
        return {
            "error": str(e)
        }


tavily_client = TavilyClient()


@tool
def tavily_tool(query: str):
    """
    Search internet using tavily
    """

    response = tavily_client.search(query=query)

    results = []

    for item in response["results"][:3]:

        results.append({
            "title": item["title"],
            "content": item["content"],
            "url": item["url"]
        })

    return results


tools = [arxiv_tool, wikipedia_tool, tavily_tool]


# =========================
# LLM
# =========================

from langchain_groq import ChatGroq
llm = ChatGroq(model='llama-3.3-70b-versatile',temperature=0)

llm_with_tools = llm.bind_tools(tools)


# =========================
# CHATBOT NODE
# =========================

def chatbot(state: AgentState):

    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages": [response]
    }


# =========================
# TOOL EXECUTION NODE
# =========================

def tool_node(state: AgentState):

    last_message = state["messages"][-1]

    arxiv_data = state.get("arxiv_results", [])
    wiki_data = state.get("wikipedia_results", [])
    tavily_data = state.get("tavily_results", [])

    for tool_call in last_message.tool_calls:

        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name == "arxiv_tool":

            result = arxiv_tool.invoke(tool_args)

            arxiv_data.append(result)

        elif tool_name == "wikipedia_tool":

            result = wikipedia_tool.invoke(tool_args)

            wiki_data.append(result)

        elif tool_name == "tavily_tool":

            result = tavily_tool.invoke(tool_args)

            tavily_data.append(result)

    return {
        "arxiv_results": arxiv_data,
        "wikipedia_results": wiki_data,
        "tavily_results": tavily_data
    }


# =========================
# CONDITIONAL EDGE
# =========================

def should_continue(state: AgentState):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END


# =========================
# GRAPH
# =========================

graph_builder = StateGraph(AgentState)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")

graph_builder.add_conditional_edges(
    "chatbot",
    should_continue
)

graph_builder.add_edge("tools", END)

graph = graph_builder.compile()


# =========================
# RUN
# =========================

initial_state = {
    "messages": [
        HumanMessage(content="Find research papers on transformers")
    ],
    "arxiv_results": [],
    "wikipedia_results": [],
    "tavily_results": []
}

result = graph.invoke(initial_state)

print("\nARXIV RESULTS:\n")
print(result["arxiv_results"])

print("\nWIKIPEDIA RESULTS:\n")
print(result["wikipedia_results"])

print("\nTAVILY RESULTS:\n")
print(result["tavily_results"])

HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=transformers&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=3)

In [7]:
initial_state = {
    "messages": [
        HumanMessage(content="Latest news in AI")
    ],
    "arxiv_results": [],
    "wikipedia_results": [],
    "tavily_results": []
}

result = graph.invoke(initial_state)

print("\nARXIV RESULTS:\n")
print(result["arxiv_results"])

print("\nWIKIPEDIA RESULTS:\n")
print(result["wikipedia_results"])

print("\nTAVILY RESULTS:\n")
print(result["tavily_results"])


ARXIV RESULTS:

[]

WIKIPEDIA RESULTS:

[]

TAVILY RESULTS:

[[{'title': 'Artificial Intelligence - Latest AI News and Analysis - WSJ.com', 'content': 'The latest artificial intelligence news coverage focusing on the technology, tools and the companies building AI technology.', 'url': 'https://www.wsj.com/tech/ai'}, {'title': 'Artificial Intelligence (Generative) Resources: Latest News on AI', 'content': 'We give AI users and leaders at enterprises the practical examples, tools and knowledge they need to thrive in the fast-changing AI era.', 'url': 'https://guides.library.georgetown.edu/ai/news'}, {'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth', 'content': 'AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.', 'url': 'https://www.artificialintelligence-news.com/'}]]


In [20]:
result["tavily_results"][0]

[{'title': 'Artificial Intelligence - Latest AI News and Analysis - WSJ.com',
  'content': 'The latest artificial intelligence news coverage focusing on the technology, tools and the companies building AI technology.',
  'url': 'https://www.wsj.com/tech/ai'},
 {'title': 'Artificial Intelligence (Generative) Resources: Latest News on AI',
  'content': 'We give AI users and leaders at enterprises the practical examples, tools and knowledge they need to thrive in the fast-changing AI era.',
  'url': 'https://guides.library.georgetown.edu/ai/news'},
 {'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
  'content': 'AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.',
  'url': 'https://www.artificialintelligence-news.com/'}]

# Implementing in proepr way


In [1]:
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from operator import add
from langchain.tools import tool

from tavily import TavilyClient
import arxiv
import wikipedia

c:\Users\prath\miniconda3\envs\langgraph311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class AgentState(TypedDict):
    messages: Annotated[list,add_messages]

    arxiv_results: Annotated[list,add]
    wikipedia_results: Annotated[list,add]
    tavily_results: Annotated[list,add]

In [3]:
@tool
def arxiv_tool(query: str):
    """
    Search research papers form arxive
    """

    client = arxiv.Client(
        page_size=3,
        delay_seconds=3,
        num_retries=3
    )

    search = arxiv.Search(
        query=query,
        max_results=3
    )

    papers = []

    for result in client.results(search):

        papers.append({
            "title":result.title,
            "summary":result.summary[:300],
            "pdf_url":result.pdf_url
        })

    return papers

@tool
def wiki_tool(query):
    """ 
    Search wikipedia information  
    """

    try:
        summary = wikipedia.summary(query, sentences=3)

        return{
            "query":query,
            "summary":summary[:300]
        }
    except Exception as e:
        return{
            "error":str(e)
        }

tavily_client = TavilyClient()

@tool
def tavily_tool(query):
    """ 
    Search internet using tavily
    """

    response = tavily_client.search(query=query)

    results = []

    for item in response['results'][:3]:

        results.append({
            "title": item["title"],
            "content": item["content"],
            "url": item["url"]            
        })
    return results

In [4]:
tools = [arxiv_tool,wiki_tool,tavily_tool]

In [5]:
llm = ChatGroq(model='llama-3.3-70b-versatile',temperature=0)
llm_with_tool = llm.bind_tools(tools)

In [6]:
def chatbot(state:AgentState):

    response = llm_with_tool.invoke(state['messages'])

    return {
        'messages':[response]
    }

In [7]:
def tool_node(state:AgentState):

    resposne = state['messages'][-1]

    arxiv_data = []
    wiki_data = []
    tavily_data = []


    for tool_call in resposne.tool_calls:

        tool_name = tool_call['name']
        tool_args = tool_call['args']

        if tool_name == "arxiv_tool":
            result = arxiv_tool.invoke(tool_args)

            arxiv_data.append(result)
        elif tool_name == "wiki_tool":
            result = wiki_tool.invoke(tool_args)
            wiki_data.append(result)
        else:
            result = tavily_tool.invoke(tool_args)
            tavily_data.append(result)

    return {
        "arxiv_results": arxiv_data,
        "wikipedia_results": wiki_data,
        "tavily_results": tavily_data
    }

In [8]:
def should_continue(state: AgentState):
    last_message = state['messages'][-1]

    if last_message.tool_calls:
        return "tools"
    
    return END

In [9]:
graph_builder = StateGraph(AgentState)
graph_builder.add_node("chatbot",chatbot)
graph_builder.add_node("tools",tool_node)

graph_builder.add_edge(START,"chatbot")

graph_builder.add_conditional_edges("chatbot",should_continue)

graph_builder.add_edge("tools", END)

In [12]:
from langgraph.checkpoint.memory import MemorySaver
from langchain.messages import HumanMessage
memory = MemorySaver()

graph = graph_builder.compile(checkpointer=memory)

In [ ]:
initial_state = {
    "messages": [
        HumanMessage(content="Attention is All You need paper")
    ]
}
thread_id = "user-1"
result = graph.invoke(initial_state,
                      config={"configurable": {"thread_id": thread_id}})

print("\nARXIV RESULTS:\n")
print(result["arxiv_results"])

print("\nWIKIPEDIA RESULTS:\n")
print(result["wikipedia_results"])

print("\nTAVILY RESULTS:\n")
result["tavily_results"]
# print(len(result['messages']))


ARXIV RESULTS:

[[{'title': 'Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet', 'summary': 'The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the', 'pdf_url': 'https://arxiv.org/pdf/2105.02723v1'}, {'title': '"All You Need" is Not All You Need for a Paper Title: On the Origins of a Scientific Meme', 'summary': "The 2017 paper ''Attention Is All You Need'' introduced the Transformer architecture-and inadvertently spawned one of machine learning's most persistent naming conventions. We analyze 717 arXiv preprints containing ''All You Need'' in their titles (2009-2025), finding exponential growth ($R^2$ > 0.9", 'pdf_url': 'https://arxiv.org/pdf/2512.19700v1'}, {'title': 'Quit When You Can: Efficient E

[[{'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.',
   'url': 'https://www.artificialintelligence-news.com/'},
  {'title': 'AI News | Latest Headlines and Developments | Reuters',
   'content': 'Explore the latest artificial intelligence news with Reuters - from AI breakthroughs and technology trends to regulation, ethics, business and global',
   'url': 'https://www.reuters.com/technology/artificial-intelligence/'},
  {'title': 'Artificial Intelligence - Latest AI News and Analysis - WSJ.com',
   'content': 'The latest artificial intelligence news coverage focusing on the technology, tools and the companies building AI technology.',
   'url': 'https://www.wsj.com/tech/ai'}],
 [{'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'AI News delivers the latest 

In [ ]:
result['arxiv_results']

[[{'title': 'Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet',
   'summary': 'The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the',
   'pdf_url': 'https://arxiv.org/pdf/2105.02723v1'},
  {'title': '"All You Need" is Not All You Need for a Paper Title: On the Origins of a Scientific Meme',
   'summary': "The 2017 paper ''Attention Is All You Need'' introduced the Transformer architecture-and inadvertently spawned one of machine learning's most persistent naming conventions. We analyze 717 arXiv preprints containing ''All You Need'' in their titles (2009-2025), finding exponential growth ($R^2$ > 0.9",
   'pdf_url': 'https://arxiv.org/pdf/2512.19700v1'},
  {'title': 'Quit When You Can: Efficient Ev

In [33]:
result

{'messages': [HumanMessage(content='Latest news in AI', additional_kwargs={}, response_metadata={}, id='f955d1eb-70c1-4c0e-8552-1672611a2388'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '43wv62a83', 'function': {'arguments': '{"query":"latest news in AI"}', 'name': 'tavily_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 314, 'total_tokens': 333, 'completion_time': 0.04999176, 'completion_tokens_details': None, 'prompt_time': 0.015653862, 'prompt_tokens_details': None, 'queue_time': 0.055991098, 'total_time': 0.065645622}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3221-ce44-79c2-825a-f0f269c11077-0', tool_calls=[{'name': 'tavily_tool', 'args': {'query': 'latest news in AI'}, 'id': '43wv62a83', 'type': 'tool_call'}], invalid_tool_calls=[], usage_meta

In [ ]:
import requests
response = requests.get(result['arxiv_results'][0][0]['pdf_url'])

In [ ]:
with open("paper.pdf", "wb") as f:
    f.write(response.content)

In [54]:
import requests
import re

def clean_filename(title):
    return re.sub(r'[\\/*?:"<>|]', "", title)

for papers in result["arxiv_results"]:
    for paper in papers:
        pdf_url = paper["pdf_url"]
        response = requests.get(pdf_url)

        filename = "papers/" + clean_filename(paper["title"]) + ".pdf"

        with open(filename, "wb") as f:
            f.write(response.content)